# Preprocessing Configuration Demo — Airline Sentiment Clustering

This Colab-ready notebook demonstrates the repository's modular P0, P1 and P2 preprocessing configurations and several controlled variants.

**Scope disclaimer:** these preprocessing choices were designed and audited for the airline-tweet sentiment-clustering context. Re-audit corpus-specific assumptions before reuse in another domain.

The raw `text` column is always retained.

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Download / update the project code from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/frankie-chew-nus/sentiment-clustering-ml.git"
REPO_DIR = "/content/sentiment-clustering-ml"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git switch main
    !git pull origin main

%cd {REPO_DIR}
!git branch --show-current
!git log -1 --oneline

## 3. Install dependencies

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q -r requirements-dev.txt
!python -m spacy download en_core_web_sm -q

## 4. Load preprocessing code and configurations

In [ ]:
import sys, importlib
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import config, lexical_normalization, preprocessing
importlib.reload(config)
importlib.reload(lexical_normalization)
importlib.reload(preprocessing)

from config import P0_CONFIG, P1_CONFIG, P2_CONFIG
from preprocessing import preprocess_text

## 5. Load the raw CSV from Google Drive

Edit `INPUT_CSV` if your Drive path differs.

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_CSV = "/content/drive/My Drive/MSI5001 project/EDA/Tweets-redacted.csv"
assert Path(INPUT_CSV).exists(), f"File not found: {INPUT_CSV}"

df = pd.read_csv(INPUT_CSV)
print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df[["text"]].head())

## 6. Compare P0 / P1 / P2 on representative tweets

In [ ]:
examples = [
    "@AmericanAir I won't fly American Airlines again!!! 😡",
    "This is #badcustomerservice",
    "Delayed &amp; nobody is helping",
    "The flights were delayed and I am very disappointed!!!",
    "The service was really very good 😊",
    "10/10 service 😊",
    "-9/10 experience",
]

for text in examples:
    print("RAW:", text)
    print("P0 :", preprocess_text(text, P0_CONFIG))
    print("P1 :", preprocess_text(text, P1_CONFIG))
    print("P2 :", preprocess_text(text, P2_CONFIG))
    print("-" * 90)

## 7. Define controlled configuration variants

In [ ]:
P2_KEEP_STOPWORDS = {**P2_CONFIG, "remove_stopwords": False}
P2_NO_LEMMA = {**P2_CONFIG, "lemmatize": False}
P2_NO_HASHTAG_SPLIT = {**P2_CONFIG, "hashtag_segmentation_mode": "none"}
P1_PLUS_LEMMA = {**P1_CONFIG, "lemmatize": True, "spacy_model": "en_core_web_sm"}

VARIANTS = {
    "P0": P0_CONFIG,
    "P1": P1_CONFIG,
    "P2": P2_CONFIG,
    "P2_keep_stopwords": P2_KEEP_STOPWORDS,
    "P2_no_lemma": P2_NO_LEMMA,
    "P2_no_hashtag_split": P2_NO_HASHTAG_SPLIT,
    "P1_plus_lemma": P1_PLUS_LEMMA,
}

demo_text = "I am not very disappointed with the delayed flights #badcustomerservice"
print("RAW:", demo_text)
for name, cfg in VARIANTS.items():
    print(f"{name:22} -> {preprocess_text(demo_text, cfg)}")

## 8. Export helper

Each export is self-contained: it writes a processed CSV **and** a JSON record of the exact configuration used. Run only the export(s) you need.

In [ ]:
import json
from pprint import pprint

OUTPUT_DIR = Path("/content/drive/My Drive/MSI5001 project/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def export_variant(config_name, run_config, output_filename):
    export_df = df.copy()
    export_df["processed_text"] = (
        export_df["text"].fillna("").astype(str)
        .apply(lambda text: preprocess_text(text, run_config))
    )

    output_csv = OUTPUT_DIR / output_filename
    config_json = OUTPUT_DIR / f"{config_name}_config.json"

    export_df.to_csv(output_csv, index=False)
    with open(config_json, "w") as f:
        json.dump(run_config, f, indent=2)

    print("Configuration:", config_name)
    print("Rows exported:", len(export_df))
    print("CSV saved:", output_csv)
    print("Config saved:", config_json)
    pprint(run_config)
    display(export_df[["text", "processed_text"]].head(10))
    return export_df

## 9. Export one variation at a time

These subsections are independent. P2 can take longer because spaCy lemmatization and stopword removal are enabled.

### 9.1 P0 — basic cleaning

In [ ]:
p0_df = export_variant("p0", {**P0_CONFIG}, "Tweets-processed-p0.csv")

### 9.2 P1 — text normalization

In [ ]:
p1_df = export_variant("p1", {**P1_CONFIG}, "Tweets-processed-p1.csv")

### 9.3 P2 — finalized lexical normalization

In [ ]:
p2_df = export_variant("p2", {**P2_CONFIG}, "Tweets-processed-p2.csv")

### 9.4 P2 with stopwords kept

In [ ]:
p2_keep_stopwords_df = export_variant(
    "p2_keep_stopwords",
    {**P2_KEEP_STOPWORDS},
    "Tweets-processed-p2_keep_stopwords.csv",
)

### 9.5 P2 without lemmatization

In [ ]:
p2_no_lemma_df = export_variant(
    "p2_no_lemma",
    {**P2_NO_LEMMA},
    "Tweets-processed-p2_no_lemma.csv",
)

### 9.6 P2 without hashtag segmentation

In [ ]:
p2_no_hashtag_split_df = export_variant(
    "p2_no_hashtag_split",
    {**P2_NO_HASHTAG_SPLIT},
    "Tweets-processed-p2_no_hashtag_split.csv",
)

### 9.7 P1 + lemmatization only

In [ ]:
p1_plus_lemma_df = export_variant(
    "p1_plus_lemma",
    {**P1_PLUS_LEMMA},
    "Tweets-processed-p1_plus_lemma.csv",
)

## 10. List saved configuration records

No prior `CONFIG_NAME` variable is required. Each Section 9 export saves its own configuration automatically.

In [ ]:
config_files = sorted(OUTPUT_DIR.glob("*_config.json"))
if not config_files:
    print("No saved configuration files found yet. Run any Section 9 export first.")
else:
    print("Saved configuration records:")
    for path in config_files:
        print(" -", path.name)

## Methodological reminders

1. Keep the raw `text` column; do not overwrite it.
2. Structural/style features and VADER should normally use raw text.
3. Sentiment labels/confidence are reserved for post-clustering evaluation only.
4. P0/P1/P2 are validated reference configurations, not proof of better clustering.
5. The configurations are tailored to the airline-tweet corpus.
6. Record the exact configuration used for every processed export.